# 01 · Coverage of FAA OE/AAA filings

What the Stage 1 backfill actually holds: how many cases per region and week, what kinds of structures, how
complete each region-year is (distinct `asnSequence` / max `asnSequence`), and where filings landed in the
last 12 months. Data comes from `data/parquet/cases.parquet`, built by `py -m faa_oe backfill` / `ingest`.

In [ ]:
from datetime import timedelta

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio
import polars as pl

from faa_oe import coverage as cov
from faa_oe import viz

viz.apply_style()
pio.renderers.default = "notebook_connected"
pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_width_chars(160)

cases = cov.load_cases().with_columns(cov.structure_group(), cov.status_group())
first, last = cases["date_entered"].min(), cases["date_entered"].max()
full_weeks = (pl.col("week") >= first) & (pl.col("week") <= last - timedelta(days=6))
print(f"{cases.height:,} cases · date_entered {first} → {last}")
cases.group_by("asn_type", "case_type").len().sort("asn_type", "case_type")

## Sanity checks
`asn_type` is the ASN suffix (the sequence series); `case_type` is the `<caseType>` tag. The OE endpoint also
serves on-airport NRA-series cases tagged `caseType=OE`, so the two differ. Below: `year` (the API path parameter and the `<year>` tag) versus the ASN prefix year and the `dateEntered` year,
plus the full list of status codes seen (looking for withdrawn / terminated cases).

In [ ]:
cases.select(
    year_eq_asn_year=(pl.col("year") == pl.col("asn_year")).mean(),
    asn_year_eq_entered_year=(pl.col("asn_year") == pl.col("date_entered").dt.year()).mean(),
    n_mismatch=(pl.col("asn_year") != pl.col("date_entered").dt.year()).sum(),
)

In [ ]:
cov.status_counts(cases)

## Weekly filings by region (OE-series ASNs)
Off-airport OE-series cases only — the population tower filings live in. One panel per ASN region, each on its own y-scale; WTE/WTW are the wind-turbine series. The current partial
week is dropped.

In [ ]:
oe = cases.filter(pl.col("asn_type") == "OE")
weekly_region = cov.weekly_counts(oe, by="asn_region").filter(full_weeks)
regions = sorted(weekly_region["asn_region"].unique())

fig, axes = viz.small_multiples(len(regions), ncols=3, sharex=True)
for ax, region in zip(axes, regions):
    d = weekly_region.filter(pl.col("asn_region") == region)
    ax.plot(d["week"], d["n"], color=viz.SERIES[0], linewidth=1)
    ax.set_title(region, fontsize=10)
    ax.set_ylim(bottom=0)
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
fig.suptitle("OE filings per week, by FAA region", x=0.01, ha="left", fontweight="semibold")
fig.tight_layout()

## Weekly filings by structure type (OE-series ASNs)
Coarse type = text before `$` in `structureType`; anything wind-related is folded into WINDMILL and gets its own
panel. The eight largest groups are shown, the rest fold into OTHER.

In [ ]:
top = oe["structure_group"].value_counts().sort("count", descending=True)["structure_group"].head(8).to_list()
oe_grouped = oe.with_columns(
    pl.when(pl.col("structure_group").is_in(top)).then(pl.col("structure_group")).otherwise(pl.lit("OTHER")).alias("structure_group")
)
weekly_type = cov.weekly_counts(oe_grouped, by="structure_group").filter(full_weeks)
order = [g for g in top if g != "WINDMILL"] + ["OTHER", "WINDMILL"]

fig, axes = viz.small_multiples(len(order), ncols=3, sharex=True)
for ax, group in zip(axes, order):
    d = weekly_type.filter(pl.col("structure_group") == group)
    color = viz.SERIES[1] if group == "WINDMILL" else viz.SERIES[0]
    ax.plot(d["week"], d["n"], color=color, linewidth=1)
    ax.set_title(group, fontsize=10)
    ax.set_ylim(bottom=0)
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
fig.suptitle("OE filings per week, by coarse structure type", x=0.01, ha="left", fontweight="semibold")
fig.tight_layout()

## Completeness by region and year
Share of issued ASN sequence numbers present in the API. Holes are cases the API no longer serves (withdrawn,
terminated, purged) — so low completeness flags survivorship risk, not just missing pulls.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), layout="constrained")
for ax, asn_type in zip(axes, ["OE", "NRA"]):
    m = cov.completeness_matrix(cases, asn_type)
    years = m.columns[1:]
    grid = m.select(years).to_numpy()
    im = ax.imshow(grid, cmap=viz.SEQUENTIAL, vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(years)), years, rotation=45)
    ax.set_yticks(range(m.height), m["asn_region"].to_list())
    ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)
    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            v = grid[i, j]
            if v == v:
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7, color="white" if v > 0.6 else viz.INK)
    ax.set_title(f"{asn_type}-series ASNs: distinct / max asnSequence")
fig.colorbar(im, ax=axes, shrink=0.8, label="completeness");

In [ ]:
cov.completeness(cases).filter(pl.col("asn_type") == "OE").pivot(
    on="asn_region", index="asn_year", values="n_cases"
).sort("asn_year")

## Where filings landed in the last 12 months (non-wind, OE-series)
Tower and monopole filings (the telecom candidates for Stage 2) in blue, all of them; other non-wind structures in
gray, randomly thinned to keep the notebook light.

In [ ]:
cutoff = cases["date_entered"].max() - timedelta(days=365)
recent = (
    oe.filter(
        (pl.col("date_entered") > cutoff)
        & (pl.col("structure_group") != "WINDMILL")
        & ~pl.col("asn_region").is_in(["WTE", "WTW"])
        & pl.col("latitude").is_not_null()
    )
    .with_columns(
        kind=pl.when(pl.col("structure_type").str.starts_with("TOWER") | (pl.col("structure_type") == "POLE$MONO"))
        .then(pl.lit("Tower / monopole"))
        .otherwise(pl.lit("Other structure"))
    )
    .select("asn", pl.col("latitude", "longitude").round(4), "kind", "structure_type", "sponsor")
)
towers = recent.filter(pl.col("kind") == "Tower / monopole")
others = recent.filter(pl.col("kind") != "Tower / monopole")
others = others.sample(min(others.height, 20_000), seed=0)
print(f"{recent.height:,} non-wind OE-series filings since {cutoff}: {towers.height:,} tower/monopole (all shown), "
      f"{others.height:,} of the rest shown")

fig = px.scatter_map(
    pl.concat([others, towers]).to_pandas(),
    lat="latitude",
    lon="longitude",
    color="kind",
    color_discrete_map={"Tower / monopole": viz.SERIES[0], "Other structure": viz.OTHER},
    hover_name="asn",
    hover_data={"structure_type": True, "sponsor": True, "latitude": False, "longitude": False, "kind": False},
    zoom=3,
    center={"lat": 39, "lon": -97},
    height=620,
    map_style="carto-positron",
)
fig.update_traces(marker={"size": 4, "opacity": 0.6})
fig.update_layout(margin=dict(l=0, r=0, t=40, b=0), title="Non-wind OE filings, last 12 months",
                  legend=dict(title=None, orientation="h", y=1.02, x=0))
fig